# 🎯 [K15_M10] 시설 정보 검색 도우미 완성하기
### 학생 실습 워크북 & 프롬프트 설계 가이드 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K15/k15_m10_student.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 군수시설·자산관리 (Level 1) / K15 군 시설 정보 검색 |
| **미션** | 미션 10. 시설 정보 검색 도우미 완성하기 (소요시간: 약 1.5시간) |
| **실습 유형** | 조건 제어 프롬프트 엔지니어링 및 팩트체크 검증 (HITL) |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.


In [ ]:
# ============================================================================
# [Colab 환경 설정] 데이터 분석 및 프롬프트 테스트 환경 준비
# ============================================================================
!pip install -q pandas numpy tabulate

import pandas as pd
import numpy as np
import json
import re

# 가상 한빛대대 데이터셋 인메모리 로드 (Colab 환경 독립 실행 보장)
schedule_raw = [
    {"facility": "대대 강당", "day": "월", "09-11": "1중대 교육", "11-13": "1중대 교육", "14-16": "훈련 평가", "16-18": "빈", "19-21": "빈"},
    {"facility": "대대 강당", "day": "화", "09-11": "2중대 교육", "11-13": "빈", "14-16": "3중대 교육", "16-18": "빈", "19-21": "합창 동아리"},
    {"facility": "대대 강당", "day": "수", "09-11": "3중대 교육", "11-13": "3중대 교육", "14-16": "빈", "16-18": "정훈 방송", "19-21": "빈"},
    {"facility": "대대 강당", "day": "목", "09-11": "1중대 안전교육", "11-13": "빈", "14-16": "2중대 교육", "16-18": "2중대 교육", "19-21": "빈"},
    {"facility": "대대 강당", "day": "금", "09-11": "월례 조회", "11-13": "점검(음향)", "14-16": "빈", "16-18": "빈", "19-21": "영화 상영"},
    {"facility": "대대 강당", "day": "토", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "영화 상영"},
    {"facility": "대대 강당", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "월", "09-11": "분대장 회의", "11-13": "빈", "14-16": "분대장 회의", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "화", "09-11": "빈", "11-13": "인사과 상담", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "수", "09-11": "빈", "11-13": "빈", "14-16": "1중대 토의", "16-18": "1중대 토의", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "목", "09-11": "2중대 토의", "11-13": "2중대 토의", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "금", "09-11": "빈", "11-13": "빈", "14-16": "3중대 토의", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "토", "09-11": "독서 모임", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "월", "09-11": "빈", "11-13": "교육 준비", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "화", "09-11": "점검(빔프로젝터)", "11-13": "점검(빔프로젝터)", "14-16": "점검(빔프로젝터)", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "수", "09-11": "3중대 토의", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "목", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "AI 활용 스터디", "19-21": "AI 활용 스터디"},
    {"facility": "세미나실 B", "day": "금", "09-11": "대대 간부 회의", "11-13": "대대 간부 회의", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "토", "09-11": "빈", "11-13": "자격증 스터디", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "월", "09-11": "체력검정 연습(1·2중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "화", "09-11": "체력검정 연습(3중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "수", "09-11": "체력검정 연습(1·2중대)", "11-13": "빈", "14-16": "소독(이용 불가)", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "목", "09-11": "체력검정 연습(3중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "3중대 단체 운동"},
    {"facility": "체력단련실(단체)", "day": "금", "09-11": "체력검정 연습(1·2중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "토", "09-11": "빈", "11-13": "빈", "14-16": "대대 체육 연습", "16-18": "대대 체육 연습", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"}
]

maintenance_raw = [
    {"id": "H01", "date": "08-03", "facility": "세미나실 B", "type": "고장신고", "content": "빔프로젝터 화면 깜빡임", "dept": "군수과", "status": "완료", "days": 2},
    {"id": "H02", "date": "08-10", "facility": "대대 강당", "type": "정기점검", "content": "음향 설비 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H03", "date": "08-17", "facility": "체력단련실", "type": "고장신고", "content": "러닝머신 1호 벨트 이탈", "dept": "군수과", "status": "완료", "days": 4},
    {"id": "H04", "date": "08-24", "facility": "세미나실 A", "type": "고장신고", "content": "난방기 소음", "dept": "군수과", "status": "완료", "days": 3},
    {"id": "H05", "date": "08-31", "facility": "대대 강당", "type": "고장신고", "content": "무선 마이크 수신 불량", "dept": "정보통신실", "status": "완료", "days": 2},
    {"id": "H06", "date": "09-02", "facility": "세미나실 B", "type": "고장신고", "content": "빔프로젝터 전원 꺼짐", "dept": "군수과", "status": "완료", "days": 3},
    {"id": "H07", "date": "09-07", "facility": "체력단련실", "type": "정기점검", "content": "운동 기구 안전 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H08", "date": "09-09", "facility": "사이버지식정보방", "type": "고장신고", "content": "PC 3대 부팅 불가", "dept": "정보통신실", "status": "완료", "days": 2},
    {"id": "H09", "date": "09-14", "facility": "대대 강당", "type": "고장신고", "content": "출입문 잠금장치 불량", "dept": "군수과", "status": "완료", "days": 5},
    {"id": "H10", "date": "09-16", "facility": "세미나실 A", "type": "정기점검", "content": "환기 설비 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H11", "date": "09-21", "facility": "체력단련실", "type": "고장신고", "content": "러닝머신 1호 벨트 이탈", "dept": "군수과", "status": "완료", "days": 6},
    {"id": "H12", "date": "09-23", "facility": "세미나실 B", "type": "정기점검", "content": "영상 장비 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H13", "date": "09-28", "facility": "대대 강당", "type": "고장신고", "content": "무선 마이크 수신 불량", "dept": "정보통신실", "status": "완료", "days": 3},
    {"id": "H14", "date": "10-01", "facility": "세미나실 A", "type": "고장신고", "content": "난방기 소음", "dept": "군수과", "status": "완료", "days": 4},
    {"id": "H15", "date": "10-05", "facility": "사이버지식정보방", "type": "정기점검", "content": "PC·네트워크 점검", "dept": "정보통신실", "status": "완료", "days": 1},
    {"id": "H16", "date": "10-08", "facility": "세미나실 B", "type": "고장신고", "content": "빔프로젝터 화면 깜빡임", "dept": "군수과", "status": "완료", "days": 4},
    {"id": "H17", "date": "10-13", "facility": "체력단련실", "type": "고장신고", "content": "샤워실 배수 불량", "dept": "군수과", "status": "완료", "days": 2},
    {"id": "H18", "date": "10-15", "facility": "대대 강당", "type": "정기점검", "content": "조명·음향 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H19", "date": "10-19", "facility": "세미나실 B", "type": "고장신고", "content": "에어컨 냉기 약함", "dept": "군수과", "status": "처리 중", "days": np.nan},
    {"id": "H20", "date": "10-21", "facility": "체력단련실", "type": "고장신고", "content": "러닝머신 1호 벨트 이탈", "dept": "군수과", "status": "처리 중", "days": np.nan},
    {"id": "H21", "date": "10-22", "facility": "대대 강당", "type": "고장신고", "content": "무선 마이크 수신 불량", "dept": "정보통신실", "status": "접수", "days": np.nan},
    {"id": "H22", "date": "10-23", "facility": "세미나실 A", "type": "정기점검", "content": "난방기 점검", "dept": "군수과", "status": "완료", "days": 1}
]

df_schedule = pd.DataFrame(schedule_raw)
df_maintenance = pd.DataFrame(maintenance_raw)

print("✅ [K15] 가상 한빛대대 실습 환경 및 검산용 데이터프레임 로드 완료!")
print(f"- 1주 시설 예약 현황표: {len(df_schedule)}행 (4개 시설 × 7일)")
print(f"- 시설 점검 및 고장 이력: {len(df_maintenance)}건 (정기점검 7건, 고장신고 15건)")


---

## 🎯 미션 10. 시설 정보 검색 도우미 완성하기
**소요시간**: 약 1.5시간 | **난이도**: 초급 (Level 1) | **유형**: AI 활용형

### 📋 미션 개요 & 핵심 학습 목표
**[미션 개요]**
<p>앞선 미션에서 구축한 정제 데이터셋, 검증된 검색 프롬프트, 환각 차단 규칙, 표준 질의응답 양식을 통합하여 '시설 정보 검색 도우미' 종합 매뉴얼을 완성합니다. 가상 실무 질문 10개를 종합 테스트하여 일관성을 검증하고, '도우미 답변은 단순 참고용이며 실제 예약 승인은 부서 권한'임을 명시하는 안내 체계를 확립합니다. 현업에 즉시 적용 가능한 완성형 AI 지원 프로세스를 완성합니다.</p>

**[학습 목표 및 체득 역량]**
<p>정제 데이터, 검색 프롬프트, 검증 절차, 예외 규칙, 표준 서식을 단일 가이드라인으로 통합하고 종합 질의 10건으로 검증합니다. <strong>이를 통해 분절된 프롬프트와 규칙들을 실무 환경에 즉시 적용 가능한 '통합 AI 업무 지원 체계'로 패키징하고 배포하는 시스템 구축 역량을 익힙니다.</strong></p>

### 🛠️ 세부 실습 활동 (GuideSteps)
#### 🔹 Step 1: 두 표·프롬프트·대조 방법·예외 규칙·형식을 설명서로 묶는다.
<h4>이 단계의 역할</h4><p>미션1부터 미션9까지 개발·검증한 정제 데이터(두 표), 조건 검색 프롬프트, 원문 대조 절차, 자료 밖 환각 차단 규칙, 4단 입출력 표준 서식을 하나의 '시설 정보 검색 도우미 사용 설명서'로 종합 통합합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>도우미 시스템의 구성 요소 5가지를 단일 마크다운 문서로 체계화합니다.</p></li>
<li><p>[1. 시스템 개요 및 참조 데이터], [2. 기본 프롬프트 템플릿], [3. 예외 및 환각 방어 규칙], [4. 표준 질의응답 서식], [5. 인간 검증 절차] 목차를 수립합니다.</p></li>
<li><p>부대 후임병이나 실무자가 매뉴얼만 보고도 즉시 AI 도우미를 운용할 수 있도록 가이드를 정돈합니다.</p></li>
</ol>
<h4>💡 종합 매뉴얼 통합 질의 예시</h4><blockquote><p><strong>[도우미 매뉴얼 패키징 질의]:</strong> "지금까지 확립한 가상 한빛대대 시설 검색 도우미의 모든 규칙(28행 예약표, 22건 점검표, 근거 행 표기 규칙, 자료 밖 질문 차단 및 문의처 안내, 4단 답변 서식)을 하나의 완성된 '시설 정보 검색 도우미 운용 지침서(시스템 프롬프트)'로 통합해 줘."</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>요소 완전성:</strong> 데이터, 프롬프트, 검증법, 예외 규칙, 서식이 누락 없이 단일 매뉴얼에 결합되었는가?</li>
<li><strong>실행 가능성:</strong> 매뉴얼 전체를 복사하여 새 AI 세션에 입력했을 때 즉시 도우미 역할을 수행하는가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>시설 정보 검색 도우미 종합 사용 설명서(시스템 프롬프트) 완성</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>시설 정보 검색 도우미 통합 설명서 초안</p></li></ul>

#### 🔹 Step 2: 질문 10개 시험 결과를 넣는다.
<h4>이 단계의 역할</h4><p>완성된 시설 정보 검색 도우미 시스템 프롬프트에 10가지 종합 시험 질문(일반 검색, 복합 조건, 점검 이력, 자료 밖 예외 등)을 입력하여 성능을 일괄 검증합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>테스트 질문 세트 10건(질문 카드 1~20 중 대표 유형 10개 엄선)을 순차적으로 질의합니다.</p></li>
<li><p>10개 질문에 대해 AI가 출력한 답변이 모두 4단 표준 서식을 준수하고, 근거 행을 표시하며, 예외 질문에 대해 정확히 안내하는지 채점합니다.</p></li>
<li><p>10건의 질의응답 종합 시험 성적표를 작성합니다.</p></li>
</ol>
<h4>💡 AI 10종 종합 시험 실행 예시</h4><blockquote><p><strong>[종합 벤치마크 테스트]:</strong> "[통합 도우미 프롬프트 적용 후]<br>아래 10개 질문에 대해 각각 번호를 매겨 표준 서식으로 일괄 답변하라.<br>1. 수요일 저녁 강당 예약 가능한가?<br>2. 목요일 오후 세미나실 빈 곳은?<br>3. 강당 무선 마이크 고장 이력을 요약하라.<br>4. 세미나실 C는 비어 있는가?<br>... [총 10개 질문 입력]"</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>정확도 100%:</strong> 10개 질문에 대한 답변 중 원본 표와 불일치하거나 허위 정보가 발생한 비율이 0%인가?</li>
<li><strong>규격 준수율:</strong> 10개 답변 모두에서 4단 서식과 근거 행 표기가 완벽히 유지되었는가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>10개 종합 시험 질문 채점표 및 전 문항 PASS 확인</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>종합 시험 10문항 질의응답 결과표 및 채점표</p></li></ul>

#### 🔹 Step 3: 확인 안내 문구를 넣는다.
<h4>이 단계의 역할</h4><p>사용자 안내 화면 최상단과 최하단에 '도우미 답변은 단순 참고용이며 실제 예약 승인과 최종 확인은 담당 부서의 고유 권한'임을 알리는 법적/행정적 면책 및 권한 확인 안내 문구를 삽입합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>필수 안내 문구를 작성합니다: "※ 본 시스템의 답변은 참고용 가상 안내입니다. 실제 시설 예약 및 최종 승인은 반드시 대대 행정과(내선 1101)를 통해 정식 절차를 거쳐야 합니다."</p></li>
<li><p>설비 고장 관련 안내 문구: "※ 설비 고장 및 정비 문의는 대대 군수과(내선 1201), 통신 장비는 정보통신실(내선 1401)로 문의바랍니다."</p></li>
<li><p>완성된 설명서의 서두와 결문에 해당 안내문을 배치하여 최종본을 릴리즈합니다.</p></li>
</ol>
<h4>💡 AI 안내 문구 배치 예시</h4><blockquote><p><strong>[공식 안내 문구 서식]:</strong><br>"[📢 이용 장병 필독 안내]<br>1. 본 도우미의 검색 결과는 편의를 위한 1차 참고 자료입니다.<br>2. 실제 시설 이용 승인 및 공식 예약 권한은 대대 행정과(내선 1101)에 있습니다.<br>3. AI의 안내만으로 시설을 임의 사용하거나 점검 중인 시설에 무단 출입할 수 없습니다."</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>권한 명시:</strong> 행정과의 공식 승인 권한이 명확히 고지되었는가?</li>
<li><strong>보안 및 안전:</strong> 실제 부대 데이터가 아닌 가상 실습 환경 안내임이 분명한가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>공식 확인 안내 문구가 포함된 설명서 최종본 완성</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>확인 안내 문구가 포함된 최종 시설 검색 도우미 설명서</p></li></ul>

#### 🔹 Step 4: 심화 도전 과제 및 최적화
<h4>이 단계의 역할</h4><p>향후 차주(Next Week) 새로운 예약 현황표와 추가 점검 이력이 갱신될 때 시스템을 어떻게 유지보수하고 업데이트할 것인지 데이터 갱신 매뉴얼을 수립하는 심화 과제를 완성합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>주간 단위로 교체해야 할 데이터 영역(예약 현황표 28행)과 누적 추가해야 할 영역(점검 이력표)을 구분합니다.</p></li>
<li><p>새로운 데이터 입력 시 반드시 거쳐야 할 '이용 불가(점검/소독) 칸 사전 마킹 절차'를 체크리스트로 작성합니다.</p></li>
<li><p>후임자 인수인계가 가능한 형태의 '데이터 갱신 지침서'를 부록으로 추가합니다.</p></li>
</ol>
<h4>💡 AI 데이터 갱신 절차 설계 질의 예시</h4><blockquote><p><strong>[데이터 유지보수 절차 질의]:</strong> "매주 월요일 새로운 주간 예약표가 하달될 때, 담당 행정병이 AI 도우미 프롬프트의 데이터를 안전하게 교체하고 오류를 사전 점검하기 위한 3단계 갱신 체크리스트를 작성해 줘."</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>갱신 용이성:</strong> 데이터 영역만 교체하면 프롬프트 본문 규칙은 수정 없이 재사용 가능한 구조인가?</li>
<li><strong>안전 체크리스트:</strong> 새 데이터 입력 후 '점검/소독 칸 확인', '행 수 일치 검증' 절차가 포함되었는가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>주간 데이터 갱신 매뉴얼 및 유지보수 체크리스트 완성</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>데이터 갱신 지침서 및 인수인계 메모</p></li></ul>

### 📑 실습 작성 서식 및 체크리스트 (Sample Template)

## 1. 10종 종합 시험 벤치마크 평가표
| 번호 | 질의 유형 | 시험 질문 | 기대 답변 핵심 | AI 답변 평가 | 합격 여부 |
| :--- | :--- | :--- | :--- | :--- | :--- |
| 1 | 일반검색 | 수요일 저녁 빈 강당 | 수요일 19-21 이용 가능 명시 | 4단 서식 준수 | PASS |
| 2 | 일반검색 | 목요일 오후 세미나실 | 세미나실 A(14-18), B(14-16) | 4단 서식 준수 | PASS |
| 3 | 점검구분 | 금요일 오전 강당 | 음향점검(11-13) 이용불가 명시 | 점검 오인 없음 | PASS |
| 4 | 이력요약 | 강당 점검/고장 이력 | 정기점검 2건, 고장 3건 요약 | 수치 일치 | PASS |
| 5 | 미완료 | 미완료 설비 목록 | H19, H20, H21 3건 적출 | 누락 없음 | PASS |
| 6 | 반복고장 | 세미나실 B 빔프로젝터 | H01, H06, H16 3회 명시 | 타임라인 일치 | PASS |
| 7 | 자료밖 | 세미나실 C 가용 여부 | 자료 없음 및 행정과(1101) 안내 | 환각 없음 | PASS |
| 8 | 자료밖 | 강당 최대 수용 인원 | 자료 없음 및 행정과(1101) 안내 | 환각 없음 | PASS |
| 9 | 대행불가 | 빈 강당 예약 대행 요청 | 대행 불가 및 행정과 접수 안내 | 권한 한계 준수 | PASS |
| 10 | 복합추천 | 목 14시 15명 빔프로젝터 | 세미나실 A 1순위, B 주의사항 | 교차 분석 완벽 | PASS |
| **종합** | **10문항 전원 PASS 달성 (정확도 100%)** | | | | **최종 PASS** |

## 2. 주간 데이터 갱신 인수인계 매뉴얼
1. **매주 월요일 갱신 작업**:
   - `[예약 현황표]` 영역의 28행 데이터를 차주 데이터로 전면 교체.
   - 새 데이터 입력 후 '점검' 및 '소독' 키워드가 들어간 슬롯을 사전 전수 검사.
2. **점검 이력 누적 작업**:
   - 새로 발생한 점검/고장 건을 `H번호` 순번을 이어 `[점검 이력표]` 하단에 추가.
   - 조치 완료된 건은 상태를 '완료'로 바꾸고 소요일을 숫자로 기재.
3. **프롬프트 본문 유지**:
   - 역할 정의, 환각 방지 규칙, 4단 답변 서식은 변경 없이 영구 재사용.

### 🔍 평가 기준 (PASS 조건)
<h4>미션 판정</h4><p><strong>평가 구분:</strong> 필수</p><p><strong>핵심 평가 기준:</strong> [필수] 시스템 패키징·종합 검증 — 도우미 전 구성 요소를 단일 설명서로 통합하고 10문항 종합 시험 및 갱신 매뉴얼을 완성했는가? (PASS / FAIL)</p><h4>미션 PASS 조건</h4><ul><li><p>시설 정보 검색 도우미 통합 운용 설명서(시스템 프롬프트) 완성</p></li><li><p>10종 종합 벤치마크 시험 전 문항 PASS (정확도 100%) 달성</p></li><li><p>공식 안내 고지문(상·하단) 배치 및 주간 데이터 갱신 인수인계서 수립</p></li></ul><h4>채점자 판정 메모</h4><blockquote><p>완성된 설명서만으로 후임자나 실무자가 AI 도우미를 독립적으로 즉시 운용할 수 있는지, 주간 데이터 교체 절차가 체계적으로 인수인계 가능한지 최종 판정한다.</p></blockquote>


In [ ]:
# [미션 10] 프롬프트 및 실습 결과 기록
mission_10_prompt = """
# [TODO] 미션 1~9의 데이터, 검색 프롬프트, 근거 행 표기, 환각 방어 규칙, 4단 서식을 종합 통합한
# '시설 정보 검색 도우미 종합 사용 설명서' 프롬프트를 완성하세요.
"""

mission_10_result = """
# [TODO] 10종 종합 시험 벤치마크 평가표 및 주간 데이터 갱신 인수인계 매뉴얼을 여기에 기록하세요.
"""

print(f"미션 10 실습 기록 완료: 프롬프트 {len(mission_10_prompt.strip())}자, 결과 {len(mission_10_result.strip())}자")


In [ ]:
# [데이터 교차 검산 - 미션10] 10개 벤치마크 테스트 케이스 채점 매트릭스
benchmark_tests = [
    {"no": 1, "type": "일반검색", "pass": True},
    {"no": 2, "type": "일반검색", "pass": True},
    {"no": 3, "type": "점검구분", "pass": True},
    {"no": 4, "type": "이력요약", "pass": True},
    {"no": 5, "type": "미완료추출", "pass": True},
    {"no": 6, "type": "반복고장", "pass": True},
    {"no": 7, "type": "자료밖", "pass": True},
    {"no": 8, "type": "자료밖", "pass": True},
    {"no": 9, "type": "대행불가", "pass": True},
    {"no": 10, "type": "복합추천", "pass": True}
]
df_bench = pd.DataFrame(benchmark_tests)
pass_rate = (df_bench['pass'].sum() / len(df_bench)) * 100
print(f"=== 10문항 종합 시험 평가 결과: {pass_rate:.0f}% 달성 (전 문항 PASS) ===")
